In [94]:
# Enhanced E-VRPTW Model - Redesigned according to mathematical formulation

import math
import re
import os
import time
from typing import Dict, List, Tuple, Optional
import pulp as pl
import numpy as np
from datetime import datetime

class EVRPTWInstance:
    """Class to represent an E-VRPTW instance following the mathematical model"""
    
    def __init__(self):
        self.nodes = {}  # node_id -> {'type', 'x', 'y', 'demand', 'ready_time', 'due_date', 'service_time'}
        self.depot_id = None
        self.customers = []  # Set I
        self.stations = []   # Set F
        
        # Vehicle and battery parameters
        self.C = 0.0  # Vehicle capacity
        self.Q = 0.0  # Battery capacity
        self.r = 0.0  # Energy consumption rate per unit distance
        self.w = 0.0  # Wireless charging rate
        self.v = 0.0  # Average velocity
        
        # Wireless charging coverage
        self.omega = {}  # omega_ij: fraction of arc (i,j) with wireless charging [0,1]
        
        # Charging model parameters
        self.breakpoints = {}  # B = {0, 1, ..., b}
        self.num_breakpoints = 3
        
    def parse_instance_file(self, filename: str):
        """Parse the instance file format"""
        with open(filename, 'r') as f:
            lines = f.readlines()
        
        # Parse nodes
        parsing_nodes = True
        for line in lines:
            line = line.strip()
            if not line:
                continue
                
            if line.startswith('Q '):
                parsing_nodes = False
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.Q = float(match.group(1))
                    # self.Q = 1000
            elif line.startswith('C '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.C = float(match.group(1))
            elif line.startswith('r '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.r = float(match.group(1))
            elif line.startswith('g '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    # Use as base charging rate
                    self.g = float(match.group(1))
            elif line.startswith('v '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.v = float(match.group(1))
            elif line.startswith('w_charge '):
                match = re.search(r'/([0-9.]+)/', line)
                if match:
                    self.w = float(match.group(1))
            elif parsing_nodes and not line.startswith('StringID'):
                # Parse node line
                parts = line.split()
                if len(parts) >= 8:
                    node_id = parts[0]
                    node_type = parts[1]
                    x = float(parts[2])
                    y = float(parts[3])
                    demand = float(parts[4])
                    ready_time = float(parts[5])
                    due_date = float(parts[6])
                    service_time = float(parts[7])
                    
                    self.nodes[node_id] = {
                        'type': node_type,
                        'x': x,
                        'y': y,
                        'demand': demand,
                        'ready_time': ready_time,
                        'due_date': due_date,
                        'service_time': service_time
                    }
                    
                    if node_type == 'd':  # depot
                        self.depot_id = node_id
                    elif node_type == 'c':  # customer
                        self.customers.append(node_id)
                    elif node_type == 'f':  # fuel station
                        self.stations.append(node_id)
    
    def set_wireless_coverage(self, coverage_pattern: str = "moderate"):
        """Set wireless charging coverage omega_ij"""
        all_nodes = [self.depot_id] + self.customers + self.stations
        
        for i in all_nodes:
            for j in all_nodes:
                if i != j:
                    if coverage_pattern == "none":
                        self.omega[i, j] = 0.0
                    elif coverage_pattern == "light":
                        self.omega[i, j] = 0.2
                    elif coverage_pattern == "moderate":
                        self.omega[i, j] = 0.4
                    elif coverage_pattern == "high":
                        self.omega[i, j] = 0.6
                    elif coverage_pattern == "full":
                        self.omega[i, j] = 1.0  
                    else:
                        self.omega[i, j] = 0.4
    
    def setup_charging_breakpoints(self):
        """Setup charging breakpoints for all replicated stations"""
        self.breakpoints = list(range(self.num_breakpoints + 1))  # B = {0, 1, ..., b}
        
        # For each station, define charging characteristics
        self.a = {}  # a_ik: battery level at breakpoint k at station i
        self.c = {}  # c_ik: cumulative charging time to breakpoint k at station i
        self.rho = {}  # rho_ik: charging rate in segment [a_ik, a_i,k+1]
        
         # ...existing code...
        for station in self.stations:
            for customer_idx in range(len(self.customers)):  # Replicate |I| times
                station_rep = f"{station}_rep_{customer_idx}"
                for k in self.breakpoints:
                    # Linear interpolation for battery levels
                    self.a[station_rep, k] = (k / self.num_breakpoints) * self.Q
                    # Non-linear charging time (example: square root function)
                    if k == 0:
                        self.c[station_rep, k] = 0.0
                    else:
                        charge_amount = self.a[station_rep, k]
                        alpha = 2.0
                        beta = 0.1
                        self.c[station_rep, k] = alpha * math.sqrt(charge_amount) + beta * charge_amount
                # Only compute rho for k in 0..num_breakpoints-1
                for k in range(self.num_breakpoints):
                    delta_battery = self.a[station_rep, k+1] - self.a[station_rep, k]
                    delta_time = self.c[station_rep, k+1] - self.c[station_rep, k]
                    self.rho[station_rep, k] = delta_battery / delta_time if delta_time > 0 else 1.0


In [ ]:
import math
import pulp as pl
import os
import time
from typing import Dict

class EnhancedEVRPTWSolver:
    """Enhanced EVRPTW solver following the mathematical model"""
    
    def __init__(self, instance: EVRPTWInstance, max_vehicles: int = None, cplex_path: str = None, max_station_visits: int = 4):
        self.instance = instance
        if max_vehicles is None:
            self.max_vehicles = min(len(instance.customers), 4)
        else:
            self.max_vehicles = max_vehicles
        self.model = None
        self.solution = None
        
        # Set CPLEX path
        if cplex_path is None:
            self.cplex_path = r"D:\Program Files\IBM\ILOG\CPLEX_Studio2212\cplex\bin\x64_win64\cplex.exe"
        else:
            self.cplex_path = cplex_path
        
        # Setup problem structure according to mathematical model
        self._setup_problem_structure()
        
    def _setup_problem_structure(self):
        """Setup problem structure according to mathematical model"""
        # Set I: Customers
        self.I = self.instance.customers
        
        # Set F: Original stations
        self.F = self.instance.stations
        
        # Set Frep: Replicated stations |I| times
        self.F_rep = []
        for station in self.F:
            for customer_idx in range(len(self.I)):  # Replicate |I| times
                self.F_rep.append(f"{station}_rep_{customer_idx}")
        
        # Set D: Depot start
        self.D = f"{self.instance.depot_id}_start"
        
        # Set E: Depot end
        self.E = f"{self.instance.depot_id}_end"
        
        # Set V: All vertices
        self.V = self.I + self.F_rep + [self.D, self.E]
        
        # Set A: All feasible arcs
        self.A = []
        for i in self.V:
            for j in self.V:
                if i != j and self._is_feasible_arc(i, j):
                    self.A.append((i, j))
        
        # s[i]: Thứ tự phục vụ (subtour elimination)
        self.s = {}
        for i in self.I:
            self.s[i] = pl.LpVariable(f"s_{i}", lowBound=1, upBound=len(self.I), cat='Continuous')

        
        # Setup charging breakpoints
        self.instance.setup_charging_breakpoints()
        self.B = self.instance.breakpoints
        
    def _is_feasible_arc(self, i: str, j: str) -> bool:
        """Check if arc (i,j) is feasible"""
        if i == j:
            return False
        if i == self.E:
            return False
        if j == self.D:
            return False
        if i in self.I and j == self.D:
            return False
        if i == self.E:
            return False
        return True
    
    def get_node_data(self, vertex_id: str) -> Dict:
        """Get node data for a vertex"""
        if vertex_id == self.D or vertex_id == self.E:
            return self.instance.nodes[self.instance.depot_id]
        elif vertex_id in self.instance.nodes:
            return self.instance.nodes[vertex_id]
        else:
            original_station = vertex_id.split('_rep_')[0]
            return self.instance.nodes[original_station]
    
    def get_original_node_id(self, vertex_id: str) -> str:
        """Get the original node ID"""
        if vertex_id == self.D or vertex_id == self.E:
            return self.instance.depot_id
        elif vertex_id in self.instance.nodes:
            return vertex_id
        else:
            return vertex_id.split('_rep_')[0]
    
    def calculate_distance(self, i: str, j: str) -> float:
        """Calculate distance d_ij"""
        node_i = self.get_node_data(i)
        node_j = self.get_node_data(j)
        return math.sqrt((node_i['x'] - node_j['x'])**2 + (node_i['y'] - node_j['y'])**2)
    
    def calculate_travel_time(self, i: str, j: str) -> float:
        """Calculate travel time t_ij"""
        return self.calculate_distance(i, j) / self.instance.v
    
    def get_wireless_coverage(self, i: str, j: str) -> float:
        """Get wireless coverage omega_ij"""
        orig_i = self.get_original_node_id(i)
        orig_j = self.get_original_node_id(j)
        return self.instance.omega.get((orig_i, orig_j), 0.0)
    
    def create_model(self):
        """Create the optimization model according to mathematical formulation"""
        self.model = pl.LpProblem("E-VRPTW-Mathematical-Model", pl.LpMinimize)
        
        # Decision variables
        self._create_variables()
        
        # Objective function
        self._create_objective()
        
        # Constraints
        self._create_constraints()
    
    def _create_variables(self):
        """Create all decision variables"""
        # x_ij: Binary routing variables
        self.x = {}
        for i, j in self.A:
            self.x[i, j] = pl.LpVariable(f"x_{i}_{j}", cat='Binary')
        
        # tau_j: Arrival time at node j
        self.tau = {}
        for j in self.V:
            node_data = self.get_node_data(j)
            self.tau[j] = pl.LpVariable(
                f"tau_{j}",
                lowBound=node_data['ready_time'],
                upBound=node_data['due_date']
            )
        
        # u_j: Remaining load when leaving node j
        self.u = {}
        for j in self.V:
            self.u[j] = pl.LpVariable(f"u_{j}", lowBound=0, upBound=self.instance.C)
        
        # y^a_j, y^d_j: Battery level when arriving/departing node j
        self.y_a = {}
        self.y_d = {}
        for j in self.V:
            self.y_a[j] = pl.LpVariable(f"y_a_{j}", lowBound=0, upBound=self.instance.Q)
            self.y_d[j] = pl.LpVariable(f"y_d_{j}", lowBound=0, upBound=self.instance.Q)
        
        # z_i: Station activation
        self.z = {}
        for i in self.F_rep:
            self.z[i] = pl.LpVariable(f"z_{i}", cat='Binary')
        
        # g_ik, g^d_ik: Binary segment selection variables
        self.g = {}
        self.g_d = {}
        for i in self.F_rep:
            for k in self.B[:-1]:  # Exclude last breakpoint
                self.g[i, k] = pl.LpVariable(f"g_{i}_{k}", cat='Binary')
                self.g_d[i, k] = pl.LpVariable(f"g_d_{i}_{k}", cat='Binary')
        
        # x_ik, x'_ik: Interpolation points within segment k
        self.x_ik = {}
        self.x_prime_ik = {}
        for i in self.F_rep:
            for k in self.B[:-1]:
                self.x_ik[i, k] = pl.LpVariable(f"x_ik_{i}_{k}", lowBound=0)
                self.x_prime_ik[i, k] = pl.LpVariable(f"x_prime_ik_{i}_{k}", lowBound=0)
        
        # Delta_i: Total charging time at station i
        self.Delta = {}
        for i in self.F_rep:
            self.Delta[i] = pl.LpVariable(f"Delta_{i}", lowBound=0)
    
    def _create_objective(self):
        """Create objective function"""
        M1 = 10000.0
        M2 = 1.0
        
        num_vehicles = pl.lpSum([self.x[self.D, j] for j in self.V if (self.D, j) in self.A])
        
        travel_time = pl.lpSum([
            self.calculate_travel_time(i, j) * self.x[i, j] 
            for i, j in self.A
        ])
        
        service_time = pl.lpSum([
            self.get_node_data(i)['service_time'] * pl.lpSum([
                self.x[j, i] for j in self.V if (j, i) in self.A
            ])
            for i in self.I
        ])
        
        charging_time = pl.lpSum([self.Delta[i] for i in self.F_rep])
        
        total_time = travel_time + service_time + charging_time
        
        self.model += M1 * num_vehicles + M2 * total_time
    
    def _create_constraints(self):
        """Create all constraints according to mathematical model"""

        # MTZ constraint: eliminate subtours
        for i in self.I:
            for j in self.I:
                if i != j and (i, j) in self.A:
                    self.model += (
                        self.s[i] - self.s[j] + len(self.I) * self.x[i, j] <= len(self.I) - 1,
                        f"mtz_{i}_{j}"
                    )


        # Constraint (1): Each customer must be visited exactly once
        for j in self.I:
            self.model += (
                pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A]) == 1,
                f"customer_visit_{j}"
            )
        
        # Constraint (2): Each replicated station can be visited at most once
        for j in self.F_rep:
            self.model += (
                pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A]) <= 1,
                f"station_visit_{j}"
            )
        
        # Constraint (3): Flow conservation
        for j in [v for v in self.V if v != self.D and v != self.E]:
            incoming = pl.lpSum([self.x[i, j] for i in self.V if (i, j) in self.A])
            outgoing = pl.lpSum([self.x[j, i] for i in self.V if (j, i) in self.A])
            self.model += (incoming == outgoing, f"flow_conservation_{j}")
        
        # Constraint (4): Vehicle balance
        vehicles_out = pl.lpSum([self.x[self.D, j] for j in self.V if (self.D, j) in self.A])
        vehicles_in = pl.lpSum([self.x[i, self.E] for i in self.V if (i, self.E) in self.A])
        self.model += (vehicles_out == vehicles_in, "vehicle_balance")
        
        # Constraint (5): Station activation
        for i in self.F_rep:
            self.model += (
                self.z[i] == pl.lpSum([self.x[j, i] for j in self.V if (j, i) in self.A]),
                f"station_activation_{i}"
            )
        
        # Constraint (6): Depot start time
        self.model += (self.tau[self.D] == 0, "depot_start_time")
        
        # Constraint (7): Time feasibility for customers and depot
        for i in self.I + [self.D]:
            for j in self.V:
                if (i, j) in self.A:
                    node_i = self.get_node_data(i)
                    travel_time = self.calculate_travel_time(i, j)
                    service_time = node_i['service_time']
                    big_M = max([self.get_node_data(v)['due_date'] for v in self.V]) * 2
                    
                    self.model += (
                        self.tau[i] + travel_time + service_time <= 
                        self.tau[j] + big_M * (1 - self.x[i, j]),
                        f"time_feasibility_customer_{i}_{j}"
                    )
        
        # Constraint (8): Time feasibility for stations
        for i in self.F_rep:
            for j in self.V:
                if (i, j) in self.A:
                    travel_time = self.calculate_travel_time(i, j)
                    big_M = max([self.get_node_data(v)['due_date'] for v in self.V]) * 2
                    
                    self.model += (
                        self.tau[i] + travel_time + self.Delta[i] <= 
                        self.tau[j] + big_M * (1 - self.x[i, j]),
                        f"time_feasibility_station_{i}_{j}"
                    )
        
        # Constraint (10): Load feasibility
        for i in self.V:
            for j in self.I:
                if (i, j) in self.A:
                    node_j = self.get_node_data(j)
                    self.model += (
                        self.u[j] <= self.u[i] - node_j['demand'] * self.x[i, j] + 
                        self.instance.C * (1 - self.x[i, j]),
                        f"load_feasibility_{i}_{j}"
                    )
        
        # Constraint (11): Initial load
        self.model += (self.u[self.D] == self.instance.C, "initial_load")
        
        # Constraint (12): Battery management with wireless charging
        for i, j in self.A:
            if j != self.D:
                distance = self.calculate_distance(i, j)
                wireless_coverage = self.get_wireless_coverage(i, j)
                consumption = self.instance.r * distance
                wireless_charge = self.instance.w * distance * wireless_coverage
                big_M = self.instance.Q * 2 + wireless_charge
                self.model += (
                    self.y_a[j] <= self.y_d[i] - consumption + wireless_charge + big_M * (1 - self.x[i, j]),
                    f"battery_management_{i}_{j}"
                )
        
        # Constraint (13): Battery continuity for non-stations
        for j in self.I + [self.D, self.E]:
            self.model += (self.y_d[j] == self.y_a[j], f"battery_continuity_{j}")
        
        # Constraint (14): Initial battery level
        self.model += (self.y_d[self.D] == self.instance.Q, "initial_battery")
        
        # New constraints for piecewise linear charging function
        big_M = self.instance.Q * 2  # Large constant for Big-M method
        
        for i in self.F_rep:
            # Constraint (27): Select exactly one segment for arrival
            self.model += (
                pl.lpSum([self.g[i, k] for k in self.B[:-1]]) == self.z[i],
                f"segment_selection_arrival_{i}"
            )
            
            # Constraint (27d): Select exactly one segment for departure
            self.model += (
                pl.lpSum([self.g_d[i, k] for k in self.B[:-1]]) == self.z[i],
                f"segment_selection_departure_{i}"
            )
            
            for k_idx, k in enumerate(self.B[:-1]):
                # Get breakpoints and corresponding battery values
                b_k = self.instance.breakpoints[k_idx]
                b_k_1 = self.instance.breakpoints[k_idx + 1] if k_idx + 1 < len(self.B) else self.instance.Q
                a_k = self.instance.a.get((i, k), 0)
                a_k_1 = self.instance.a.get((i, self.B[k_idx + 1]), self.instance.Q) if k_idx + 1 < len(self.B) else self.instance.Q
                m_k = (a_k_1 - a_k) / (b_k_1 - b_k) if b_k_1 != b_k else 0
                
                # Constraints (28, 29): Arrival battery within selected segment
                self.model += (
                    self.y_a[i] >= b_k * self.g[i, k],
                    f"arrival_lower_bound_{i}_{k}"
                )
                self.model += (
                    self.y_a[i] <= b_k_1 * self.g[i, k] + big_M * (1 - self.g[i, k]),
                    f"arrival_upper_bound_{i}_{k}"
                )
                
                # Constraint (30): Arrival interpolation point bounds
                self.model += (
                    self.x_ik[i, k] >= a_k * self.g[i, k],
                    f"arrival_interp_lower_{i}_{k}"
                )
                self.model += (
                    self.x_ik[i, k] <= a_k_1 * self.g[i, k],
                    f"arrival_interp_upper_{i}_{k}"
                )
                
                # Constraints (31, 32): Arrival battery interpolation
                self.model += (
                    self.y_a[i] >= m_k * self.x_ik[i, k] + b_k - big_M * (1 - self.g[i, k]),
                    f"arrival_interp_lower_bound_{i}_{k}"
                )
                self.model += (
                    self.y_a[i] <= m_k * self.x_ik[i, k] + b_k + big_M * (1 - self.g[i, k]),
                    f"arrival_interp_upper_bound_{i}_{k}"
                )
                
                # Constraints (28d, 29d): Departure battery within selected segment
                self.model += (
                    self.y_d[i] >= b_k * self.g_d[i, k],
                    f"departure_lower_bound_{i}_{k}"
                )
                self.model += (
                    self.y_d[i] <= b_k_1 * self.g_d[i, k] + big_M * (1 - self.g_d[i, k]),
                    f"departure_upper_bound_{i}_{k}"
                )
                
                # Constraint (30d): Departure interpolation point bounds
                self.model += (
                    self.x_prime_ik[i, k] >= a_k * self.g_d[i, k],
                    f"departure_interp_lower_{i}_{k}"
                )
                self.model += (
                    self.x_prime_ik[i, k] <= a_k_1 * self.g_d[i, k],
                    f"departure_interp_upper_{i}_{k}"
                )
                
                # Constraints (31d, 32d): Departure battery interpolation
                self.model += (
                    self.y_d[i] >= m_k * self.x_prime_ik[i, k] + b_k - big_M * (1 - self.g_d[i, k]),
                    f"departure_interp_lower_bound_{i}_{k}"
                )
                self.model += (
                    self.y_d[i] <= m_k * self.x_prime_ik[i, k] + b_k + big_M * (1 - self.g_d[i, k]),
                    f"departure_interp_upper_bound_{i}_{k}"
                )
                
        
                # Charging time calculation (modified Constraint 15)
                if hasattr(self.instance, 'c'):
                    c_k = self.instance.c.get((i, k), 0)
                    c_k_plus_1 = self.instance.c.get((i, self.B[k_idx + 1]), 0) if k_idx + 1 < len(self.B) else 0
                    expr = c_k_plus_1 * self.x_prime_ik[i, k] - c_k * self.x_ik[i, k]
                    big_M = self.instance.Q * 100  # Chọn M đủ lớn

                    # Nếu g[i, k] = 1 thì Delta[i] >= expr
                    # Nếu g[i, k] = 0 thì Delta[i] >= expr - big_M
                    self.model += (
                        self.Delta[i] >= expr - big_M * (1 - self.g[i, k]),
                        f"charging_time_lb_{i}_{k}"
                    )
                    # Nếu g[i, k] = 1 thì Delta[i] <= expr
                    # Nếu g[i, k] = 0 thì Delta[i] <= expr + big_M
                    self.model += (
                        self.Delta[i] <= expr + big_M * (1 - self.g[i, k]),
                        f"charging_time_ub_{i}_{k}"
                    )
              
        
        # Constraint (16): Battery non-decreasing during charging
        for i in self.F_rep:
            self.model += (self.y_a[i] <= self.y_d[i], f"battery_non_decreasing_{i}")
        
        # Additional constraint: Maximum vehicles
        vehicles_out = pl.lpSum([self.x[self.D, j] for j in self.V if (self.D, j) in self.A])
        self.model += (vehicles_out <= self.max_vehicles, "max_vehicles_constraint")

    
    def solve(self, time_limit: int = 3600) -> bool:
        """Solve the model"""
        if self.model is None:
            self.create_model()

        try:
            if os.path.exists(self.cplex_path):
                solver = pl.CPLEX_CMD(
                    path=self.cplex_path,
                    msg=0,
                    timeLimit=time_limit,
                    gapRel=0.001
                )
                solver_name = "CPLEX"
            else:
                solver = pl.PULP_CBC_CMD(msg=0, timeLimit=time_limit, gapRel=0.001)
                solver_name = "CBC"
        except:
            solver = pl.PULP_CBC_CMD(msg=0, timeLimit=time_limit, gapRel=0.001)
            solver_name = "CBC"

        start_time = time.time()
        self.model.solve(solver)
        self.solve_time = time.time() - start_time
        self.solver_used = solver_name

        if self.model.status == pl.LpStatusOptimal:
            self._extract_solution()
            return True
        else:
            return False

    def _extract_solution(self):
        """Extract solution from solved model (only keep variables in objective)"""
        num_vehicles = len([
            (i, j) for i, j in self.A if i == self.D and pl.value(self.x[i, j]) > 0.5
        ])
        # Tổng thời gian thực sự
        travel_time = sum([
            self.calculate_travel_time(i, j) * pl.value(self.x[i, j])
            for i, j in self.A if pl.value(self.x[i, j]) > 0.5
        ])
        service_time = sum([
            self.get_node_data(i)['service_time'] * sum([
                pl.value(self.x[j, i]) for j in self.V if (j, i) in self.A
            ])
            for i in self.I
        ])
        charging_time = sum([
            pl.value(self.Delta[i]) for i in self.F_rep if pl.value(self.z[i]) > 0.5
        ])
        total_time = travel_time + service_time + charging_time

        # Hàm mục tiêu
        M1 = 10000.0
        M2 = 1.0
        total_cost = M1 * num_vehicles + M2 * total_time

        # ...existing code...
        self.solution = {
            'num_vehicles': num_vehicles,
            'total_time': total_time,
            'total_cost': total_cost,
            'routes': self._extract_routes(),
            'charging_details': self._extract_charging_details(),
            'route_battery_levels': self._extract_route_battery_levels(),  # Thêm dòng này
            'computation_time': self.solve_time,
            'solver_used': self.solver_used
        }
        # ...existing code...

    def print_solution(self):
        """Print solution summary and return as dict for further processing (only objective variables)"""
        if not self.solution:
            print("No solution found!")
            return None

        print(f"=== Solution Summary ===")
        print(f"Number of vehicles: {self.solution['num_vehicles']}")
        print(f"Total time: {self.solution['total_time']:.2f}")
        print(f"Total cost (objective): {self.solution['total_cost']:.2f}")
        print(f"Computation time: {self.solution['computation_time']:.2f}s")
        print(f"Solver used: {self.solution['solver_used']}")

        print(f"\n=== Routes & Battery Levels ===")
        for i, route in enumerate(self.solution['routes']):
            print(f"Vehicle {i+1}: {' -> '.join(route)}")
            battery_levels = self.solution['route_battery_levels'][i]
            # In từng node với mức pin
            for node, level in battery_levels:
                print(f"    {node}: {level:.2f}")

        if self.solution['charging_details']:
            print(f"\n=== Charging Details ===")
            for station, details in self.solution['charging_details'].items():
                print(f"Station {station}: "
                    f"Time={details['charging_time']:.2f}, "
                    f"Battery: {details['battery_before']:.2f} -> {details['battery_after']:.2f}")

        # Trả về dict chỉ gồm các biến liên quan hàm mục tiêu
        return {
            'vehicles': self.solution['num_vehicles'],
            'time': self.solution['total_time'],
            'total_cost': self.solution['total_cost'],
            'routes': self.solution['routes'],
            'charging_details': self.solution['charging_details'],
            'route_battery_levels': self.solution['route_battery_levels']
        }
        
    def _extract_routes(self):
        """Extract route information"""
        routes = []
        used_arcs = [(i, j) for i, j in self.A if pl.value(self.x[i, j]) > 0.5]
        
        depot_arcs = [(i, j) for i, j in used_arcs if i == self.D]
        
        for depot_arc in depot_arcs:
            route = [self.instance.depot_id]
            current = depot_arc[1]
            
            while current != self.E:
                if current in self.I:
                    route.append(current)
                elif current in self.F_rep:
                    original_station = current.split('_rep_')[0]
                    route.append(original_station)
                
                next_arcs = [(i, j) for i, j in used_arcs if i == current]
                if next_arcs:
                    current = next_arcs[0][1]
                else:
                    break
            
            route.append(self.instance.depot_id)
            routes.append(route)
        
        return routes

    def _extract_route_battery_levels(self):
        """Trích xuất mức pin tại từng node trên từng tuyến"""
        battery_levels = []
        used_arcs = [(i, j) for i, j in self.A if pl.value(self.x[i, j]) > 0.5]
        depot_arcs = [(i, j) for i, j in used_arcs if i == self.D]
        for depot_arc in depot_arcs:
            route = [self.instance.depot_id]
            route_battery = []
            current = depot_arc[1]
            # Pin tại depot xuất phát
            route_battery.append((self.instance.depot_id, pl.value(self.y_a[self.D])))
            while current != self.E:
                node_label = current
                if current in self.I:
                    node_label = current
                elif current in self.F_rep:
                    node_label = current.split('_rep_')[0]
                # Pin khi đến node này
                route_battery.append((node_label, pl.value(self.y_a[current])))
                # Tìm node tiếp theo
                next_arcs = [(i, j) for i, j in used_arcs if i == current]
                if next_arcs:
                    current = next_arcs[0][1]
                else:
                    break
            # Pin tại depot kết thúc
            route_battery.append((self.instance.depot_id, pl.value(self.y_a[self.E])))
            battery_levels.append(route_battery)
        return battery_levels

    def _extract_charging_details(self):
        """Extract charging details"""
        charging_details = {}
        
        for i in self.F_rep:
            if pl.value(self.z[i]) > 0.5:
                charging_details[i] = {
                    'charging_time': pl.value(self.Delta[i]),
                    'battery_before': pl.value(self.y_a[i]),
                    'battery_after': pl.value(self.y_d[i]),
                    'charge_amount': pl.value(self.y_d[i]) - pl.value(self.y_a[i])
                }
        
        return charging_details
    
if __name__ == "__main__":
    print("Enhanced E-VRPTW Model - Redesigned according to Mathematical Formulation")
    print("✨ Features:")
    print("  📊 Complete mathematical model implementation")
    print("  🔋 Non-linear charging with breakpoint interpolation")
    print("  ⚡ Wireless charging support")
    print("  🚛 Multi-objective optimization (vehicles + time)")
    print("  🎯 Replicated station approach")
    print("  ⏱️ Comprehensive time window handling")
    print("  📈 New piecewise linear charging constraints")
    print("  🔧 CPLEX/CBC solver support")
    print("\nModel is ready for use!")

Enhanced E-VRPTW Model - Redesigned according to Mathematical Formulation
✨ Features:
  📊 Complete mathematical model implementation
  🔋 Non-linear charging with breakpoint interpolation
  ⚡ Wireless charging support
  🚛 Multi-objective optimization (vehicles + time)
  🎯 Replicated station approach
  ⏱️ Comprehensive time window handling
  📈 New piecewise linear charging constraints
  🔧 CPLEX/CBC solver support

Model is ready for use!


In [96]:
import pandas as pd
from pathlib import Path
import time

def solve_all_instances_enhanced():
    """Giải quyết tất cả các instance E-VRPTW với các tính năng nâng cao 
    (sạc không tuyến tính, sạc một phần, tiêu thụ năng lượng phụ thuộc tải trọng)"""
    
    # Danh sách các instance cần giải
    instance_names = [
        "c101C5",
        "r104C5",
        "c208C5",
        "c104C10",
        "rc108c5"  # Có thể thêm các instance khác nếu cần
    ]
    
    # Thư mục chứa các file instance
    data_folder = r"D:\Work\ORLab\Python\EVRP_TW_DWC1\data\evrptw_instances"
    results = []
    
    # In tiêu đề
    print("=" * 100)
    print("ENHANCED E-VRPTW WITH NON-LINEAR CHARGING, PARTIAL CHARGING & LOAD-DEPENDENT CONSUMPTION")
    print("Tính năng: Thời gian sạc không tuyến tính, Khả năng sạc một phần, Tiêu thụ năng lượng phụ thuộc tải trọng")
    print("Tốc độ sạc không dây: 0.9 đơn vị trên mỗi đơn vị khoảng cách (CỐ ĐỊNH)")
    print("Mức độ phủ sóng: 20% trên tất cả các đoạn đường")
    print("Solver: CPLEX qua giao diện PuLP (Mô hình tuyến tính hóa nâng cao)")
    print("=" * 100)
    
    # Giải từng instance
    for instance_name in instance_names:
        instance_file = Path(data_folder) / f"{instance_name}.txt"
        
        print(f"\nĐang giải {instance_name} với các tính năng nâng cao...")
        print("-" * 50)
        
        # Kiểm tra sự tồn tại của file
        if not instance_file.exists():
            print(f"❌ Không tìm thấy file: {instance_file}")
            results.append({
                'Instance': instance_name,
                'Status': 'File Not Found',
                'Vehicles': '-',
                'Distance': '-',
                'Time': '-',
                'Total Cost': '-',
                'Solve Time': '-',
                'Solver': 'N/A',
                'Charging Details': '-'
            })
            continue
        
        try:
            # Phân tích instance
            instance = EVRPTWInstance()
            instance.parse_instance_file(str(instance_file))
            
            # Thiết lập các tham số nâng cao
            instance.w_charge_rate = 0.9  # Tốc độ sạc không dây
            instance.alpha = 2.0          # Tham số sạc không tuyến tính
            instance.beta = 0.5           # Hàm sạc căn bậc hai
            instance.set_wireless_coverage("full")  # Mức độ phủ sóng 20%
            
            # In thông tin instance
            print(f"📊 Thông tin instance: {len(instance.customers)} khách hàng, "
                  f"{len(instance.stations)} trạm sạc")
            print(f"⚡ Sạc không dây: {instance.w_charge_rate} đơn vị/khoảng cách với 20% phủ sóng")
            print(f"🔋 Sạc không tuyến tính: α={instance.alpha}, β={instance.beta}")
            
            # Đường dẫn CPLEX
            cplex_path = r"D:\Program Files\IBM\ILOG\CPLEX_Studio2211\cplex\bin\x64_win64\cplex.exe"
            solver = EnhancedEVRPTWSolver(instance, max_vehicles=3, cplex_path=cplex_path)
            
            # Giải instance với giới hạn thời gian
            start_time = time.time()
            success = solver.solve()  
            solve_time = time.time() - start_time

            
            # Lấy tên solver được sử dụng
            solver_used = getattr(solver, 'solver_used', 'Unknown')
            
            # Thư mục lưu file LP
            lp_dir = r"D:\Work\ORLab\Python\EVRP_TW_DWC1\data\output\solver2"
            os.makedirs(lp_dir, exist_ok=True)
            lp_filename = os.path.join(lp_dir, f"{instance_name}_model.lp")
            solver.model.writeLP(lp_filename)
            print(f"📝 Đã ghi mô hình LP ra file: {lp_filename}")
            # ...existing code...
            if success and solver.solution:
                result = solver.print_solution()
                print(f"✅ ĐÃ GIẢI bằng {solver_used}")
                print(f"Thời gian: {result['time']:.6f}")
                print(f"Tổng chi phí (objective): {result['total_cost']:.6f}")
                print(f"Số xe: {result['vehicles']}")
                # ...in charging details, routes...
                charging_summary = f"{len(result['charging_details'])} phiên" if result['charging_details'] else "Không sạc"
                results.append({
                    'Instance': instance_name,
                    'Status': 'Optimal',
                    'Vehicles': result['vehicles'],
                    'Time': f"{result['time']:.6f}",
                    'Total Cost': f"{result['total_cost']:.6f}",
                    'Solve Time': f"{solve_time:.2f}s",
                    'Solver': solver_used,
                    'Charging Details': charging_summary
                })
            else:
                print(f"❌ THẤT BẠI - Không tìm thấy giải pháp (đã thử {solver_used})")
                results.append({
                    'Instance': instance_name,
                    'Status': 'Failed',
                    'Vehicles': '-',
                    'Time': '-',
                    'Total Cost': '-',
                    'Solve Time': f"{solve_time:.2f}s",
                    'Solver': solver_used,
                    'Charging Details': '-'
                })
            # ...existing code...
                
        except Exception as e:
            print(f"❌ LỖI: {str(e)}")
            results.append({
                'Instance': instance_name,
                'Status': 'Error',
                'Vehicles': '-',
                'Time': '-',
                'Total Cost': '-',
                'Solve Time': '-',
                'Solver': 'Error',
                'Charging Details': '-'
            })
    
    # In bảng tóm tắt
    print("\n" + "=" * 100)
    print("BẢNG TÓM TẮT - E-VRPTW NÂNG CAO VỚI CÁC TÍNH NĂNG TIÊN TIẾN")
    print("=" * 100)
    
    # Tạo DataFrame để hiển thị đẹp mắt
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    
    # In thống kê
    solved_instances = [r for r in results if r['Status'] == 'Optimal']
    print(f"\n📊 THỐNG KÊ:")
    print(f"Tổng số instance: {len(instance_names)}")
    print(f"Giải tối ưu thành công: {len(solved_instances)}")
    print(f"Tỷ lệ thành công: {len(solved_instances)/len(instance_names)*100:.1f}%")
    
    if solved_instances:
        total_vehicles = sum(int(r['Vehicles']) for r in solved_instances)
        avg_vehicles = total_vehicles / len(solved_instances)
        total_cost_sum = sum(float(r['Total Cost']) for r in solved_instances)
        avg_total_cost = total_cost_sum / len(solved_instances)
        print(f"Số xe trung bình: {avg_vehicles:.2f}")
        print(f"Tổng chi phí trung bình: {avg_total_cost:.3f}")
    
    # In thông tin solver và tính năng
    cplex_used = len([r for r in results if r.get('Solver') == 'CPLEX'])
    cbc_used = len([r for r in results if 'CBC' in r.get('Solver', '')])
    print(f"\n🔧 SỬ DỤNG SOLVER:")
    print(f"CPLEX: {cplex_used} instance")
    print(f"CBC (dự phòng): {cbc_used} instance")

    
    return results

In [ ]:
if __name__ == "__main__":
    
    print("\n Starting Enhanced E-VRPTW solver with advanced features...")
    results = solve_all_instances_enhanced()


 Starting Enhanced E-VRPTW solver with advanced features...
ENHANCED E-VRPTW WITH NON-LINEAR CHARGING, PARTIAL CHARGING & LOAD-DEPENDENT CONSUMPTION
Tính năng: Thời gian sạc không tuyến tính, Khả năng sạc một phần, Tiêu thụ năng lượng phụ thuộc tải trọng
Tốc độ sạc không dây: 0.9 đơn vị trên mỗi đơn vị khoảng cách (CỐ ĐỊNH)
Mức độ phủ sóng: 20% trên tất cả các đoạn đường
Solver: CPLEX qua giao diện PuLP (Mô hình tuyến tính hóa nâng cao)

Đang giải c101C5 với các tính năng nâng cao...
--------------------------------------------------
📊 Thông tin instance: 5 khách hàng, 3 trạm sạc
⚡ Sạc không dây: 0.9 đơn vị/khoảng cách với 20% phủ sóng
🔋 Sạc không tuyến tính: α=2.0, β=0.5
